# Transformation pipelines : `Pipeline`, `ColumnTransformer`, `make_column_selector`

Carnet CS229, bloc **B1M** · appliqué à un **registre de maintenance** · **version à trous**

---

## Le problème

Un jeu de données réel n'est jamais une matrice de nombres propres. Celui-ci contient :

- des colonnes **numériques** d'échelles très différentes (des heures de vol à cinq chiffres,
  des vibrations à trois décimales) ;
- des colonnes **textuelles** qu'aucun modèle ne sait lire ;
- une colonne **ordonnée** dont l'ordre porte du sens (`mineure < moyenne < majeure < critique`) ;
- une catégorie **déguisée en nombre**, qui piège les sélecteurs automatiques ;
- des **valeurs manquantes** produites par trois mécanismes différents.

Tout cela doit être transformé avant d'atteindre le modèle — et surtout **refait à
l'identique** à chaque fold de validation croisée, puis en production. C'est le travail du
pipeline.

## Mode d'emploi

Les cellules marquées `# TODO` contiennent des `...` à remplacer. Chacune est suivie d'une
**cellule de vérification**.

**8 trous.** Comptez 2 h. Aucun téléchargement : le registre est généré.

## Ce que vous saurez faire à la fin

Assembler un objet unique qui va des données brutes à la prédiction, régler les choix de
preprocessing par recherche d'hyperparamètres, et savoir **quand** la fuite de données est un
vrai danger — ce qui, vous allez le mesurer, n'est pas toujours le cas.

---
## 1. Environnement

*Cellule complète.*

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SEED = 0
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110

print("pandas", pd.__version__, "| sklearn", __import__("sklearn").__version__)

pandas 2.3.3 | sklearn 1.7.2


---
## 2. Le registre de maintenance

Chaque ligne est une intervention. On veut prédire `retour_30j` : l'appareil est-il revenu en
atelier dans les trente jours ?

*Cellule complète.*

In [2]:
TYPES = ["A320", "A350", "B737", "B787", "ATR72"]
ATELIERS = ["Toulouse", "Hambourg", "Seville", "Bordeaux"]
INTERV = ["visite", "remplacement", "reparation", "controle", "revision"]
CRITICITE = ["mineure", "moyenne", "majeure", "critique"]
EQUIPES = ["equipe-A", "equipe-B", "equipe-C"]


def genere_registre(n=1200, seed=SEED):
    rng = np.random.default_rng(seed)

    heures_vol = rng.gamma(6, 900, n).round(0)
    cycles = (heures_vol / rng.uniform(1.6, 3.4, n)).round(0)
    jours_depuis_inspection = rng.integers(1, 400, n).astype(float)
    temperature_releve = rng.normal(62, 14, n).round(1)
    vibration_rms = rng.gamma(3, 0.55, n).round(3)
    cout_piece = rng.lognormal(6.4, 1.0, n).round(2)

    type_appareil = rng.choice(TYPES, n, p=[.30, .18, .27, .15, .10])
    atelier = rng.choice(ATELIERS, n, p=[.40, .25, .20, .15])
    type_intervention = rng.choice(INTERV, n, p=[.30, .22, .20, .18, .10])
    criticite = rng.choice(CRITICITE, n, p=[.42, .31, .19, .08])
    equipe = rng.choice(EQUIPES, n)

    code_atelier = pd.Categorical(atelier, categories=ATELIERS).codes.astype(float)
    crit_num = pd.Categorical(criticite, categories=CRITICITE, ordered=True).codes

    logit = (-2.9 + 0.00022 * heures_vol + 0.62 * crit_num + 0.45 * vibration_rms
             + 0.0021 * jours_depuis_inspection
             + 0.50 * (type_intervention == "reparation")
             - 0.40 * (type_intervention == "revision")
             + rng.normal(0, 0.45, n))
    retour = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

    df = pd.DataFrame({
        "heures_vol": heures_vol, "cycles": cycles,
        "jours_depuis_inspection": jours_depuis_inspection,
        "temperature_releve": temperature_releve, "vibration_rms": vibration_rms,
        "cout_piece": cout_piece, "code_atelier": code_atelier,
        "type_appareil": type_appareil, "atelier": atelier,
        "type_intervention": type_intervention, "criticite": criticite,
        "equipe": equipe, "retour_30j": retour,
    })

    # trois mecanismes de défaillances differents
    df.loc[rng.random(n) < 0.18, "vibration_rms"] = np.nan          # capteur HS, au hasard
    p = np.where(df["type_appareil"] == "ATR72", 0.35, 0.07)
    df.loc[rng.random(n) < p, "temperature_releve"] = np.nan        # depend de l'appareil
    p = np.where(df["type_intervention"] == "controle", 0.40, 0.05)
    df.loc[rng.random(n) < p, "cout_piece"] = np.nan                # depend du type d'acte
    df.loc[rng.random(n) < 0.06, "equipe"] = np.nan
    df.loc[rng.random(n) < 0.03, "criticite"] = np.nan
    return df


registre = genere_registre()
X = registre.drop(columns="retour_30j")
y = registre["retour_30j"]

print("registre :", registre.shape)
print(f"taux de retour sous 30 j : {y.mean():.1%}")
registre.head()

registre : (1200, 13)
taux de retour sous 30 j : 50.4%


,heures_vol,cycles,jours_depuis_inspection,temperature_releve,vibration_rms,cout_piece,code_atelier,type_appareil,atelier,type_intervention,criticite,equipe,retour_30j
0,5374.0,1739.0,362.0,90.5,0.216,287.60,0.0,B737,Toulouse,controle,moyenne,equipe-B,0
1,6599.0,3119.0,284.0,47.8,0.987,121.61,0.0,A320,Toulouse,remplacement,moyenne,equipe-A,1
2,4036.0,1662.0,132.0,81.5,0.747,540.93,0.0,A320,Toulouse,revision,critique,equipe-C,1
3,8435.0,3997.0,78.0,NaN,2.637,1836.16,3.0,B737,Bordeaux,visite,mineure,equipe-C,1
4,3736.0,1942.0,127.0,82.5,2.398,NaN,1.0,B737,Hambourg,controle,moyenne,NaN,1


---
## 3. Exploration

*Cellules complètes. Regardez-les avant de coder quoi que ce soit.*

In [3]:
info = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "manquants": X.isna().sum(),
    "% manquants": (X.isna().mean() * 100).round(1),
    "valeurs distinctes": X.nunique(),
})
info

,dtype,manquants,% manquants,valeurs distinctes
heures_vol,float64,0,0.0,1108
cycles,float64,0,0.0,1003
jours_depuis_inspection,float64,0,0.0,379
temperature_releve,float64,98,8.2,491
vibration_rms,float64,231,19.2,822
cout_piece,float64,133,11.1,1066
code_atelier,float64,0,0.0,4
type_appareil,object,0,0.0,5
atelier,object,0,0.0,4
type_intervention,object,0,0.0,5


### Les trois mécanismes de manquance

Toutes les valeurs absentes ne se valent pas, et la distinction change la façon de les
traiter (point **B1M-5**).

In [4]:
print("vibration_rms manquante, par type d'appareil :")
print((X["vibration_rms"].isna().groupby(X["type_appareil"]).mean() * 100).round(1).to_string())
print("\ntemperature_releve manquante, par type d'appareil :")
print((X["temperature_releve"].isna().groupby(X["type_appareil"]).mean() * 100).round(1).to_string())
print("\ncout_piece manquant, par type d'intervention :")
print((X["cout_piece"].isna().groupby(X["type_intervention"]).mean() * 100).round(1).to_string())

vibration_rms manquante, par type d'appareil :
type_appareil
A320     20.1
A350     19.7
ATR72    15.9
B737     19.8
B787     18.0

temperature_releve manquante, par type d'appareil :
type_appareil
A320      5.7
A350      7.9
ATR72    26.2
B737      7.5
B787      3.8

cout_piece manquant, par type d'intervention :
type_intervention
controle        39.6
remplacement     3.2
reparation       4.1
revision         2.5
visite           4.4


Trois situations différentes :

**`vibration_rms`** manque de façon uniforme, quel que soit l'appareil. Un capteur tombe en
panne sans rapport avec ce qu'il mesure. C'est du **MCAR** (*missing completely at random*) :
imputer par la médiane est sans danger.

**`temperature_releve`** manque cinq fois plus souvent sur les ATR72. L'absence dépend d'une
**autre colonne observée** — c'est du **MAR** (*missing at random*). L'imputation reste
légitime, mais le fait même que la valeur manque porte de l'information, et on pourrait vouloir
la conserver.

**`cout_piece`** manque surtout pour les contrôles, parce qu'un contrôle sans remplacement n'a
pas de coût pièce. Ce n'est pas une donnée perdue : **c'est un zéro**. L'imputer par la médiane
des autres interventions serait une erreur de modélisation, pas de statistique.

Retenez le principe : **avant d'imputer, demandez-vous pourquoi la valeur manque.** La réponse
est dans le métier, pas dans les données.

---
## 4. Séparer les colonnes par type

### Trou 1

La première chose dont un `ColumnTransformer` a besoin, c'est de savoir quelles colonnes vont
où.

In [10]:
# TODO (trou 1) - Construire deux listes de NOMS de colonnes.
#   colonnes_num : les colonnes numeriques de X
#   colonnes_cat : toutes les autres
#   Indice : X.select_dtypes(include=...) / (exclude=...) renvoie un sous-DataFrame ;
#            .columns.tolist() en extrait les noms.

colonnes_num = X.select_dtypes(include=[np.number]).columns.tolist()        # <-- a completer
colonnes_cat = X.select_dtypes(include=[object]).columns.tolist()      # <-- a completer

print("numeriques  :", colonnes_num)
print("categorielles:", colonnes_cat)

numeriques  : ['heures_vol', 'cycles', 'jours_depuis_inspection', 'temperature_releve', 'vibration_rms', 'cout_piece', 'code_atelier']
categorielles: ['type_appareil', 'atelier', 'type_intervention', 'criticite', 'equipe']


In [11]:
# Verification - ne modifiez pas cette cellule
assert isinstance(colonnes_num, list) and isinstance(colonnes_cat, list), \
    "les deux doivent etre des listes de noms, pas des DataFrames ni des Index"
assert set(colonnes_num) | set(colonnes_cat) == set(X.columns), \
    "ensemble, les deux listes doivent couvrir TOUTES les colonnes de X"
assert not (set(colonnes_num) & set(colonnes_cat)), \
    "aucune colonne ne doit figurer dans les deux listes"
assert "heures_vol" in colonnes_num and "atelier" in colonnes_cat, \
    "heures_vol est numerique, atelier est categoriel"
assert "retour_30j" not in colonnes_num + colonnes_cat, \
    "la cible ne doit pas figurer dans les features (X l'exclut deja)"
print("Trou 1 : OK")
print(f"{len(colonnes_num)} numeriques, {len(colonnes_cat)} categorielles")

Trou 1 : OK
7 numeriques, 5 categorielles


### Le piège : une catégorie déguisée en nombre

Regardez la liste des numériques. `code_atelier` s'y trouve, parce qu'il est stocké en
`float64`. Mais regardez ce qu'il contient.

In [12]:
print("code_atelier :", sorted(X["code_atelier"].unique()))
print()
print(pd.crosstab(X["code_atelier"], X["atelier"]).to_string())

code_atelier : [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0)]

atelier       Bordeaux  Hambourg  Seville  Toulouse
code_atelier                                       
0.0                  0         0        0       461
1.0                  0       312        0         0
2.0                  0         0      245         0
3.0                182         0        0         0


C'est l'atelier, encodé en entier. Le traiter comme une grandeur continue revient à affirmer
que Séville (2) est « deux fois » Hambourg (1), et que Bordeaux (3) en est le triple. Le
modèle linéaire lui attribuerait un coefficient unique, comme s'il existait un ordre et une
distance entre les ateliers.

**Aucun test automatique ne détecte cela.** `select_dtypes` lit le type de stockage, pas le
sens. C'est à vous de regarder (point **B1M-6**).

On corrige à la main. Au passage, `criticite` est mise de côté : elle est bien textuelle, mais
elle est **ordonnée**, et la section 9 lui réservera un traitement à part.

*Cellule complète.*

In [13]:
colonnes_num = [c for c in colonnes_num if c != "code_atelier"]
colonnes_cat = [c for c in colonnes_cat if c != "criticite"] + ["code_atelier"]
colonnes_ord = ["criticite"]

print("numeriques   :", colonnes_num)
print("categorielles:", colonnes_cat)
print("ordinales    :", colonnes_ord)

numeriques   : ['heures_vol', 'cycles', 'jours_depuis_inspection', 'temperature_releve', 'vibration_rms', 'cout_piece']
categorielles: ['type_appareil', 'atelier', 'type_intervention', 'equipe', 'code_atelier']
ordinales    : ['criticite']


---
## 5. Le pipeline numérique

Un `Pipeline` est une **suite d'étapes nommées**. Chaque étape sauf la dernière doit être un
*transformer* (il a `fit_transform`) ; la dernière peut être un transformer ou un modèle.

```python
Pipeline([("nom_etape_1", Transformer1()), ("nom_etape_2", Transformer2())])
```

Les noms ne sont pas décoratifs : ils servent à désigner les hyperparamètres lors du grid
search, avec la syntaxe `etape__parametre` (**double underscore**). Vous vous en servirez au
trou 8.

### Trou 2

Deux étapes pour les colonnes numériques : imputer les manquants par la **médiane**, puis
standardiser.

Pourquoi la médiane plutôt que la moyenne : `cout_piece` suit une loi log-normale avec quelques
valeurs très élevées, qui tirent la moyenne vers le haut. La médiane est robuste aux valeurs
extrêmes.

In [16]:
# TODO (trou 2) - Pipeline numerique a deux etapes.
#   Nommez-les exactement "imputer" et "scaler" (les asserts et le grid search s'en servent).
#   imputer : SimpleImputer, strategie mediane
#   scaler  : StandardScaler

pipeline_num = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])       # <-- a completer

pipeline_num

,steps,"[('imputer', ...), ('scaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [17]:
# Verification - ne modifiez pas cette cellule
assert isinstance(pipeline_num, Pipeline), "pipeline_num doit etre un objet Pipeline"
assert [n for n, _ in pipeline_num.steps] == ["imputer", "scaler"], \
    f'les etapes doivent s\'appeler "imputer" puis "scaler", pas {[n for n,_ in pipeline_num.steps]}'
assert pipeline_num.named_steps["imputer"].strategy == "median", \
    "la strategie d'imputation doit etre 'median'"
assert isinstance(pipeline_num.named_steps["scaler"], StandardScaler)

_out = pipeline_num.fit_transform(X[colonnes_num])
assert not np.isnan(_out).any(), "il reste des NaN apres imputation"
assert abs(_out.mean()) < 1e-10, "apres StandardScaler, la moyenne doit etre nulle"
assert abs(_out.std() - 1) < 1e-6, "apres StandardScaler, l'ecart-type doit valoir 1"
print("Trou 2 : OK")
print(f"  mediane de vibration_rms apprise : "
      f"{pipeline_num.named_steps['imputer'].statistics_[colonnes_num.index('vibration_rms')]:.3f}")

Trou 2 : OK
  mediane de vibration_rms apprise : 1.481


**L'ordre des étapes n'est pas interchangeable.** Imputer d'abord, standardiser ensuite. Si
vous standardisiez d'abord, `StandardScaler` calculerait sa moyenne et son écart-type en
ignorant les `NaN`, puis l'imputation injecterait une médiane dans une échelle déjà figée.
Le résultat serait incohérent.

---
## 6. Le pipeline catégoriel

### Trou 3

Deux étapes également : imputer par la valeur la plus fréquente, puis encoder en one-hot.

`handle_unknown="ignore"` est **indispensable**. Sans lui, si un fold de validation contient
une modalité absente du fold d'entraînement — un nouvel atelier, une équipe créée depuis —
l'encodeur lève une exception et tout s'arrête. Avec lui, la ligne reçoit des zéros partout
(point **BHD-10**).

In [23]:
# TODO (trou 3) - Pipeline categoriel a deux etapes.
#   Nommez-les exactement "imputer" et "encoder".
#   imputer : SimpleImputer, strategie "most_frequent"
#   encoder : OneHotEncoder avec handle_unknown="ignore"

pipeline_cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))])           # <-- a completer

pipeline_cat

,steps,"[('imputer', ...), ('encoder', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,categories,'auto'


In [24]:
# Verification - ne modifiez pas cette cellule
assert isinstance(pipeline_cat, Pipeline), "pipeline_cat doit etre un objet Pipeline"
assert [n for n, _ in pipeline_cat.steps] == ["imputer", "encoder"], \
    f'les etapes doivent s\'appeler "imputer" puis "encoder"'
assert pipeline_cat.named_steps["imputer"].strategy == "most_frequent"
assert pipeline_cat.named_steps["encoder"].handle_unknown == "ignore", \
    "handle_unknown='ignore' est indispensable : sans lui, une modalite inconnue fait planter"

_sub = X[colonnes_cat]
_enc = pipeline_cat.fit(_sub)
_t = _enc.transform(_sub)
assert _t.shape[0] == len(X), "une ligne en entree doit donner une ligne en sortie"

# une modalite jamais vue ne doit pas faire planter
_nouveau = _sub.iloc[[0]].copy()
_nouveau.loc[:, "atelier"] = "Nantes"
_r = _enc.transform(_nouveau)
assert _r.shape[1] == _t.shape[1], "le nombre de colonnes doit rester identique"
print("Trou 3 : OK")
print(f"  {len(colonnes_cat)} colonnes categorielles -> {_t.shape[1]} colonnes one-hot")
print("  une modalite inconnue ('Nantes') passe sans erreur")

Trou 3 : OK
  5 colonnes categorielles -> 21 colonnes one-hot
  une modalite inconnue ('Nantes') passe sans erreur


---
## 7. Le `ColumnTransformer`

Un `Pipeline` applique ses étapes **en série, à toutes les colonnes**. Or ici, numériques et
catégorielles demandent des traitements différents, appliqués **en parallèle** sur des
colonnes différentes.

C'est le rôle du `ColumnTransformer` :

```python
ColumnTransformer([
    ("nom_du_bloc", transformer, liste_de_colonnes),
    ...
])
```

Il applique chaque transformer à ses colonnes, puis **concatène horizontalement** les
résultats.

### Trou 4

In [25]:
# TODO (trou 4) - Assembler les deux pipelines.
#   Nommez les blocs exactement "num" et "cat".
#   Chaque bloc recoit sa liste de colonnes (colonnes_num / colonnes_cat).

preprocessing = ColumnTransformer([
    ("num", pipeline_num, colonnes_num),
    ("cat", pipeline_cat, colonnes_cat),
])# <-- a completer

preprocessing

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [26]:
# Verification - ne modifiez pas cette cellule
assert isinstance(preprocessing, ColumnTransformer)
_noms = [n for n, _, _ in preprocessing.transformers]
assert _noms == ["num", "cat"], f'les blocs doivent s\'appeler "num" puis "cat", pas {_noms}'

_M = preprocessing.fit_transform(X)
_M = _M.toarray() if hasattr(_M, "toarray") else _M
assert _M.shape[0] == len(X), "le nombre de lignes doit etre conserve"
assert not np.isnan(_M).any(), "il ne doit plus rester un seul NaN"

# criticite n'est pas encore branchee : elle ne doit PAS apparaitre
_f = preprocessing.get_feature_names_out()
assert not any("criticite" in n for n in _f), \
    "criticite est traitee a part (section 9), elle ne doit pas etre dans ce bloc"
print("Trou 4 : OK")
print(f"  {X.shape[1]} colonnes brutes -> {_M.shape[1]} colonnes numeriques")

Trou 4 : OK
  12 colonnes brutes -> 27 colonnes numeriques


In [27]:
_f

array(['num__heures_vol', 'num__cycles', 'num__jours_depuis_inspection',
       'num__temperature_releve', 'num__vibration_rms', 'num__cout_piece',
       'cat__type_appareil_A320', 'cat__type_appareil_A350',
       'cat__type_appareil_ATR72', 'cat__type_appareil_B737',
       'cat__type_appareil_B787', 'cat__atelier_Bordeaux',
       'cat__atelier_Hambourg', 'cat__atelier_Seville',
       'cat__atelier_Toulouse', 'cat__type_intervention_controle',
       'cat__type_intervention_remplacement',
       'cat__type_intervention_reparation',
       'cat__type_intervention_revision', 'cat__type_intervention_visite',
       'cat__equipe_equipe-A', 'cat__equipe_equipe-B',
       'cat__equipe_equipe-C', 'cat__code_atelier_0.0',
       'cat__code_atelier_1.0', 'cat__code_atelier_2.0',
       'cat__code_atelier_3.0'], dtype=object)

Les colonnes non mentionnées sont **supprimées**, c'est le comportement par défaut
(`remainder="drop"`). C'est pour cela que `criticite`, absente des deux listes, a disparu.

Deux autres valeurs existent : `remainder="passthrough"` laisse passer les colonnes restantes
telles quelles, et `remainder=un_transformer` leur applique un traitement commun. Le défaut
`"drop"` est le plus sûr — il vous oblige à déclarer explicitement ce que vous gardez.

---
## 8. `make_column_selector`

Lister les colonnes à la main a deux défauts : il faut recommencer dès que le schéma change,
et une colonne ajoutée en amont passe silencieusement à la trappe.

`make_column_selector` remplace la liste par une **règle**, évaluée au moment du `fit` :

```python
make_column_selector(dtype_include=np.number)       # par type
make_column_selector(pattern="^capteur_")           # par motif dans le nom
```

### Trou 5

In [31]:
# TODO (trou 5) - Refaire le ColumnTransformer avec des selecteurs au lieu de listes.
#   num : selectionner par dtype numerique
#   cat : selectionner par dtype objet
#   Memes noms de blocs ("num", "cat") et memes pipelines qu'au trou 4.

preprocessing_auto = ColumnTransformer([
    ("num", pipeline_num, make_column_selector(dtype_include=np.number)),
    ("cat", pipeline_cat, make_column_selector(dtype_include=object)),
     ]) # <-- a completer

preprocessing_auto

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [32]:
# Verification - ne modifiez pas cette cellule
assert isinstance(preprocessing_auto, ColumnTransformer)
assert [n for n, _, _ in preprocessing_auto.transformers] == ["num", "cat"]
for _n, _, _sel in preprocessing_auto.transformers:
    assert callable(_sel), f'le bloc "{_n}" doit recevoir un SELECTEUR, pas une liste de noms'

_A = preprocessing_auto.fit_transform(X)
_A = _A.toarray() if hasattr(_A, "toarray") else _A
assert _A.shape[0] == len(X)
assert not np.isnan(_A).any()
print("Trou 5 : OK")
print("  colonnes choisies par le selecteur numerique :")
print("   ", make_column_selector(dtype_include=np.number)(X))

Trou 5 : OK
  colonnes choisies par le selecteur numerique :
    ['heures_vol', 'cycles', 'jours_depuis_inspection', 'temperature_releve', 'vibration_rms', 'cout_piece', 'code_atelier']


### La contrepartie

Regardez la liste qu'affiche la cellule : **`code_atelier` est de retour côté numérique.**

Le sélecteur fait exactement ce qu'on lui demande — trier par type de stockage — et il ne peut
pas deviner qu'un `float64` représente un atelier. L'automatisation a réintroduit le bug que
vous aviez corrigé à la main à la section 4.

**La bonne pratique** : utiliser les sélecteurs pour ce qu'ils font bien — s'adapter à un
schéma qui bouge — mais **corriger le typage en amont**, pour que la règle automatique tombe
juste.

*Cellule complète : on répare à la source.*

In [33]:
X_typed = X.copy()
X_typed["code_atelier"] = X_typed["code_atelier"].astype("object")

print("avec le typage corrige, le selecteur numerique donne :")
print("  ", make_column_selector(dtype_include=np.number)(X_typed))
print("et le selecteur objet :")
print("  ", make_column_selector(dtype_include=object)(X_typed))

avec le typage corrige, le selecteur numerique donne :
   ['heures_vol', 'cycles', 'jours_depuis_inspection', 'temperature_releve', 'vibration_rms', 'cout_piece']
et le selecteur objet :
   ['code_atelier', 'type_appareil', 'atelier', 'type_intervention', 'criticite', 'equipe']


Un mot sur la robustesse : selon la version de pandas, les colonnes de texte apparaissent en
`object` ou en `str`. Si `dtype_include=object` ne renvoie rien chez vous, utilisez
`dtype_exclude=np.number`, qui désigne le complémentaire et reste juste dans tous les cas.

---
## 9. La criticité : une catégorie **ordonnée**

`mineure < moyenne < majeure < critique`. Cet ordre est réel, et le one-hot le détruirait :
il produirait quatre colonnes indépendantes, sans aucune relation entre elles.

`OrdinalEncoder` conserve l'ordre, **à condition de le lui donner explicitement**. Sans
l'argument `categories`, il trie par ordre alphabétique — ce qui donnerait ici
`critique < majeure < mineure < moyenne`, un ordre absurde.

### Trou 6

In [34]:
print("ordre alphabetique (ce qu'on NE veut PAS) :", sorted(CRITICITE))
print("ordre metier (ce qu'on veut)              :", CRITICITE)

ordre alphabetique (ce qu'on NE veut PAS) : ['critique', 'majeure', 'mineure', 'moyenne']
ordre metier (ce qu'on veut)              : ['mineure', 'moyenne', 'majeure', 'critique']


In [35]:
# TODO (trou 6) - Pipeline pour la colonne ordonnee.
#   Etapes nommees "imputer" puis "encoder".
#   imputer : SimpleImputer, strategie "most_frequent"
#   encoder : OrdinalEncoder, en lui passant l'ordre metier
#   Indice : OrdinalEncoder(categories=[...]) attend une LISTE DE LISTES,
#            une liste d'ordres par colonne traitee. Ici une seule colonne.

pipeline_ord = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OrdinalEncoder(categories=[CRITICITE]))])        # <-- a completer

pipeline_ord

,steps,"[('imputer', ...), ('encoder', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,categories,"[['mineure', 'moyenne', ...]]"


In [36]:
# Verification - ne modifiez pas cette cellule
assert isinstance(pipeline_ord, Pipeline)
assert [n for n, _ in pipeline_ord.steps] == ["imputer", "encoder"]

_enc = pipeline_ord.named_steps["encoder"]
assert isinstance(_enc, OrdinalEncoder), "l'etape 'encoder' doit etre un OrdinalEncoder"
assert _enc.categories is not None and _enc.categories != "auto", \
    "sans categories=[...], l'ordre serait alphabetique : critique < majeure < mineure"
assert list(_enc.categories[0]) == CRITICITE, \
    f"l'ordre doit etre {CRITICITE}, pas {list(_enc.categories[0])}"

_o = pipeline_ord.fit_transform(X[["criticite"]])
assert _o.shape == (len(X), 1), "une colonne ordonnee donne UNE colonne, pas quatre"
_table = {c: float(pipeline_ord.transform(pd.DataFrame({"criticite": [c]}))[0, 0])
          for c in CRITICITE}
assert list(_table.values()) == [0.0, 1.0, 2.0, 3.0], f"codage attendu 0,1,2,3 : {_table}"
print("Trou 6 : OK")
print("  codage :", _table)

Trou 6 : OK
  codage : {'mineure': 0.0, 'moyenne': 1.0, 'majeure': 2.0, 'critique': 3.0}


**Quand l'encodage ordinal est-il légitime ?** Pour un modèle **linéaire**, il impose que
l'effet soit régulier : passer de mineure à moyenne aurait le même poids que de majeure à
critique. Rien ne le garantit.

Pour un **arbre**, la question ne se pose pas : il compare des seuils, donc seul l'ordre
compte, pas l'espacement. C'est une raison de préférer l'encodage ordinal avec des modèles à
base d'arbres, et de se méfier avec des modèles linéaires.

---
## 10. Le pipeline complet

### Trou 7

Un seul objet, des données brutes à la prédiction : le `ColumnTransformer` à trois blocs, puis
le modèle.

In [48]:
# TODO (trou 7a) - ColumnTransformer a TROIS blocs : "num", "cat", "ord".
#   Utilisez les listes colonnes_num, colonnes_cat, colonnes_ord.

preprocessing_complet = ColumnTransformer([
    ("num", pipeline_num, colonnes_num),
    ("cat", pipeline_cat, colonnes_cat),
    ("ord", pipeline_ord, colonnes_ord),
])# <-- a completer


# TODO (trou 7b) - Pipeline final : etape "prep" puis etape "clf".
#   prep : le ColumnTransformer ci-dessus
#   clf  : LogisticRegression(max_iter=2000, random_state=SEED)

modele = Pipeline([("prep", preprocessing_complet), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])       # <-- a completer

modele

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [49]:
# Verification - ne modifiez pas cette cellule
assert isinstance(preprocessing_complet, ColumnTransformer)
assert [n for n, _, _ in preprocessing_complet.transformers] == ["num", "cat", "ord"], \
    'les trois blocs doivent s\'appeler "num", "cat", "ord"'
assert isinstance(modele, Pipeline)
assert [n for n, _ in modele.steps] == ["prep", "clf"], \
    'les etapes du pipeline final doivent s\'appeler "prep" puis "clf"'
assert isinstance(modele.named_steps["clf"], LogisticRegression)

modele.fit(X, y)
_p = modele.predict_proba(X)
assert _p.shape == (len(X), 2), "predict_proba doit renvoyer deux colonnes"
_noms = modele.named_steps["prep"].get_feature_names_out()
assert any("ord__criticite" in n for n in _noms), "le bloc ordinal doit apparaitre en sortie"
print("Trou 7 : OK")
print(f"  {X.shape[1]} colonnes brutes -> {len(_noms)} features")

Trou 7 : OK
  12 colonnes brutes -> 28 features


### Ce que le pipeline garantit

Le `modele` que vous venez de construire accepte **le DataFrame brut**, avec ses `NaN` et son
texte. Il n'y a plus d'étape manuelle à ne pas oublier.

Et surtout : à chaque appel de `fit`, les médianes, les modes, les moyennes et les écarts-types
sont recalculés **sur les seules données d'entraînement**. En validation croisée, cela se
produit automatiquement à chaque fold.

In [50]:
scores = cross_val_score(modele, X, y, cv=CV, scoring="roc_auc")
print(f"AUC en validation croisee : {scores.mean():.4f} +/- {scores.std():.4f}")
print("par fold :", scores.round(4))

AUC en validation croisee : 0.7032 +/- 0.0214
par fold : [0.7407 0.6964 0.7111 0.6799 0.688 ]


In [51]:
noms = modele.named_steps["prep"].get_feature_names_out()
coefs = modele.named_steps["clf"].coef_[0]
imp = pd.DataFrame({"feature": noms, "coefficient": coefs})
imp["abs"] = imp["coefficient"].abs()
print("Les 10 coefficients les plus forts :")
print(imp.sort_values("abs", ascending=False).head(10).to_string(index=False))

Les 10 coefficients les plus forts :
                          feature  coefficient      abs
                   ord__criticite     0.603403 0.603403
  cat__type_intervention_revision    -0.563024 0.563024
cat__type_intervention_reparation     0.499363 0.499363
               num__vibration_rms     0.434906 0.434906
          cat__type_appareil_B787    -0.368721 0.368721
                  num__heures_vol     0.268658 0.268658
          cat__type_appareil_A350     0.199527 0.199527
                      num__cycles     0.196046 0.196046
     num__jours_depuis_inspection     0.165052 0.165052
            cat__atelier_Toulouse    -0.147554 0.147554


In [59]:
imp.head()

,feature,coefficient,abs
0,num__heures_vol,0.268658,0.268658
1,num__cycles,0.196046,0.196046
2,num__jours_depuis_inspection,0.165052,0.165052
3,num__temperature_releve,0.040204,0.040204
4,num__vibration_rms,0.434906,0.434906


Les préfixes `num__`, `cat__` et `ord__` viennent des noms de blocs : `get_feature_names_out`
trace l'origine de chaque colonne. C'est ce qui rend un pipeline **interprétable** malgré la
transformation.

Vérifiez que les coefficients dominants correspondent bien aux facteurs qui ont servi à
générer les données : `criticite`, `vibration_rms`, `heures_vol` et le type d'intervention.

---
## 11. La fuite de données : mesurons-la

L'argument habituel en faveur des pipelines est qu'ils empêchent la **fuite de données** : si
vous imputez sur la totalité du jeu avant de découper, les folds de validation ont influencé
les médianes, et le score devient optimiste.

C'est vrai. Mais **de combien** ? Mesurons au lieu de l'affirmer.

*Cellule complète.*

In [60]:
# A) propre : tout le preprocessing est DANS le pipeline
propre = cross_val_score(modele, X, y, cv=CV, scoring="roc_auc")

# B) fuite : preprocessing applique une fois, sur TOUT le jeu, avant la CV
X_fuite = preprocessing_complet.fit_transform(X)
fuite = cross_val_score(LogisticRegression(max_iter=2000, random_state=SEED),
                        X_fuite, y, cv=CV, scoring="roc_auc")

print(f"A) pipeline propre       : AUC {propre.mean():.5f} +/- {propre.std():.5f}")
print(f"B) preprocessing en amont: AUC {fuite.mean():.5f} +/- {fuite.std():.5f}")
print(f"\necart : {fuite.mean() - propre.mean():+.5f}   (ecart-type entre folds : {propre.std():.5f})")

A) pipeline propre       : AUC 0.70322 +/- 0.02142
B) preprocessing en amont: AUC 0.70298 +/- 0.02138

ecart : -0.00024   (ecart-type entre folds : 0.02142)


**L'écart est négligeable** — de l'ordre de $10^{-4}$, deux ordres de grandeur en dessous de
l'écart-type entre folds.

Regardez même son **signe** : selon la graine, il sort positif ou négatif. Une fuite réelle
avantagerait systématiquement la version B ; un écart qui change de sens d'une exécution à
l'autre est du bruit, pas un biais.

C'est un résultat contraire au discours habituel, et il s'explique : avec $n = 1200$, retirer
un cinquième des lignes ne déplace pratiquement pas une médiane ou un mode. La fuite existe
en théorie, elle est indétectable en pratique.

### Alors quand la fuite est-elle dangereuse ?

Quand l'étape de preprocessing **regarde la cible**, ou **fait un choix** parmi beaucoup
d'options. Démonstration sur des données qui ne contiennent **aucun signal**.

In [62]:
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(0)
X_bruit = rng.normal(size=(100, 2000))      # 2000 features de bruit pur
y_bruit = rng.integers(0, 2, 100)           # cible independante
cv_b = StratifiedKFold(5, shuffle=True, random_state=0)

dans = Pipeline([("sel", SelectKBest(f_classif, k=20)),
                 ("clf", LogisticRegression(max_iter=1000))])
a = cross_val_score(dans, X_bruit, y_bruit, cv=cv_b, scoring="roc_auc")

X_sel = SelectKBest(f_classif, k=20).fit_transform(X_bruit, y_bruit)
b = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_bruit, cv=cv_b, scoring="roc_auc")

print("Donnees 100 % bruit : la verite est AUC = 0.500\n")
print(f"  selection DANS le pipeline  : AUC {a.mean():.3f}   (honnete)")
print(f"  selection AVANT la CV       : AUC {b.mean():.3f}   <- pure illusion")

Donnees 100 % bruit : la verite est AUC = 0.500

  selection DANS le pipeline  : AUC 0.540   (honnete)
  selection AVANT la CV       : AUC 0.881   <- pure illusion


Sur des données **sans aucun signal**, sélectionner les 20 meilleures features sur l'ensemble
du jeu produit un AUC de 0,88. Le modèle ne mesure que sa propre capacité à avoir choisi, parmi
2 000 colonnes de bruit, celles qui corrèlent par hasard avec la cible — y compris dans les
folds de validation.

**La règle qui en découle** :

| Étape | Fuite |
|---|---|
| Imputation par médiane ou mode, standardisation | faible, souvent négligeable |
| Sélection de features, PCA supervisée, rééchantillonnage (SMOTE) | **massive** |
| Encodage par la cible (*target encoding*) | **massive** |

Et le vrai argument pour les pipelines n'est donc pas la peur de la fuite. C'est qu'un
pipeline rend la fuite **structurellement impossible**, sans que vous ayez à vous demander, à
chaque étape, dans quelle catégorie du tableau elle tombe.

---
## 12. Régler le preprocessing par grid search

Voici la capacité que rien d'autre n'offre : les choix de preprocessing deviennent des
**hyperparamètres**, réglables comme les autres.

La syntaxe suit le chemin dans l'arborescence, avec un **double underscore** par niveau :

```
prep__num__imputer__strategy
 │     │     │        └── le parametre du SimpleImputer
 │     │     └── l'etape "imputer" du pipeline numerique
 │     └── le bloc "num" du ColumnTransformer
 └── l'etape "prep" du pipeline final
```

### Trou 8

In [68]:
# TODO (trou 8) - Construire la grille d'hyperparametres.
#   Trois axes a explorer :
#     - la strategie d'imputation numerique : "median" ou "mean"
#     - la strategie d'imputation categorielle : "most_frequent" ou "constant"
#     - la regularisation du modele : C = 0.1, 1.0 ou 10.0
#   Format : un dictionnaire {chemin: [valeurs]}
#   Indice : les chemins commencent par prep__num__, prep__cat__ et clf__

grille = {'prep__num__imputer__strategy': ["median", "mean"], 'prep__cat__imputer__strategy': ["most_frequent", "constant"], 'clf__C':[0.1, 1.0, 10.0]}       # <-- a completer

print(f"{np.prod([len(v) for v in grille.values()])} combinaisons")

12 combinaisons


In [69]:
# Verification - ne modifiez pas cette cellule
assert isinstance(grille, dict), "grille doit etre un dictionnaire"
_attendus = {"prep__num__imputer__strategy", "prep__cat__imputer__strategy", "clf__C"}
assert set(grille) == _attendus, (
    f"cles attendues :\n  {sorted(_attendus)}\ntrouvees :\n  {sorted(grille)}")
assert sorted(grille["prep__num__imputer__strategy"]) == ["mean", "median"]
assert sorted(grille["prep__cat__imputer__strategy"]) == ["constant", "most_frequent"]
assert sorted(grille["clf__C"]) == [0.1, 1.0, 10.0]

# les chemins doivent reellement designer des parametres du pipeline
_valides = modele.get_params(deep=True)
for _k in grille:
    assert _k in _valides, f"'{_k}' n'est pas un parametre de votre pipeline"
print("Trou 8 : OK  (les 3 chemins designent de vrais parametres)")

Trou 8 : OK  (les 3 chemins designent de vrais parametres)


In [70]:
gs = GridSearchCV(modele, grille, cv=CV, scoring="roc_auc", n_jobs=1)
gs.fit(X, y)

print("meilleurs parametres :")
for k, v in gs.best_params_.items():
    print(f"  {k} = {v}")
print(f"\nmeilleur AUC : {gs.best_score_:.4f}")

meilleurs parametres :
  clf__C = 0.1
  prep__cat__imputer__strategy = constant
  prep__num__imputer__strategy = mean

meilleur AUC : 0.7046


In [73]:
gs.best_params_

{'clf__C': 0.1,
 'prep__cat__imputer__strategy': 'constant',
 'prep__num__imputer__strategy': 'mean'}

In [75]:
res = pd.DataFrame(gs.cv_results_)

tab = pd.DataFrame({
    "imput. num": res["param_prep__num__imputer__strategy"],
    "imput. cat": res["param_prep__cat__imputer__strategy"],
    "C":          res["param_clf__C"],
    "AUC":        res["mean_test_score"].round(4),
    "ecart-type": res["std_test_score"].round(4),
    "rang":       res["rank_test_score"],
}).sort_values("AUC", ascending=False)

print(tab.head(6).to_string(index=False))

imput. num    imput. cat    C    AUC  ecart-type  rang
      mean      constant  0.1 0.7046      0.0225     1
    median      constant  0.1 0.7041      0.0213     2
      mean most_frequent  0.1 0.7039      0.0231     3
      mean      constant  1.0 0.7039      0.0222     4
      mean      constant 10.0 0.7038      0.0218     5
    median      constant 10.0 0.7037      0.0207     6


### Lisez ce tableau comme il faut être lu

Les six premières lignes tiennent dans un mouchoir de poche : quelques millièmes d'écart, pour
un écart-type entre folds de l'ordre de 0,02. **Ces configurations ne sont pas distinguables**
(point **BHD-12**).

La conclusion n'est donc pas « médiane bat moyenne » mais : *le choix de la stratégie
d'imputation ne change rien ici*. C'est une information utile — elle vous autorise à prendre la
plus simple et la plus défendable devant un ingénieur maintenance, sans arrière-pensée de
performance.

Un grid search qui ne départage pas est un résultat, pas un échec.

---
## 13. Ce qui ne marche pas, et pourquoi

Trois variantes qu'on vous recommandera un jour. Testons-les.

*Cellule complète.*

In [76]:
def auc(pipe):
    s = cross_val_score(pipe, X, y, cv=CV, scoring="roc_auc")
    return s.mean(), s.std()


def variante(add_indicator, ordinal):
    npipe = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=add_indicator)),
                      ("scaler", StandardScaler())])
    cpipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(handle_unknown="ignore"))])
    if ordinal:
        blocs = [("num", npipe, colonnes_num), ("cat", cpipe, colonnes_cat),
                 ("ord", pipeline_ord, colonnes_ord)]
    else:
        blocs = [("num", npipe, colonnes_num), ("cat", cpipe, colonnes_cat + colonnes_ord)]
    return Pipeline([("prep", ColumnTransformer(blocs)),
                     ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])


print(f"{'add_indicator':>15}{'criticite ordinale':>22}{'AUC':>12}{'ecart-type':>13}")
for ai in [False, True]:
    for od in [False, True]:
        m, s = auc(variante(ai, od))
        print(f"{str(ai):>15}{str(od):>22}{m:12.4f}{s:13.4f}")

  add_indicator    criticite ordinale         AUC   ecart-type
          False                 False      0.7048       0.0203
          False                  True      0.7032       0.0214
           True                 False      0.7007       0.0260
           True                  True      0.6987       0.0271


Les quatre variantes sont **indistinguables**. Quelques millièmes d'écart pour un écart-type
de 0,02.

Pourtant, les deux idées étaient bonnes sur le papier.

**L'indicateur de manquance** (`add_indicator=True`) ajoute une colonne binaire par feature
imputée, pour que le modèle sache que la valeur était absente. On a vu en section 3 que
l'absence de `temperature_releve` dépend du type d'appareil : elle *porte* de l'information.
Mais cette information, le modèle la possède déjà — `type_appareil` est dans les features.
L'indicateur est redondant.

**L'encodage ordinal de la criticité** impose un effet régulier, alors que le one-hot laisse le
modèle estimer quatre coefficients libres. Avec 1 200 lignes, il a largement de quoi les
estimer ; la contrainte n'apporte rien.

**Le principe général** : une amélioration de preprocessing ne paie que si elle apporte une
information que le modèle n'a pas, ou impose une contrainte dont il a besoin. Sinon elle ajoute
de la complexité pour rien. Et la seule façon de le savoir est de mesurer.

---
## 14. Bilan

### Les trois objets

| Objet | Rôle | Composition |
|---|---|---|
| `Pipeline` | enchaîner des étapes **en série** | liste de `(nom, transformer)`, modèle en dernier |
| `ColumnTransformer` | appliquer des traitements **en parallèle** selon la colonne | liste de `(nom, transformer, colonnes)` |
| `make_column_selector` | désigner des colonnes par **règle** plutôt que par liste | `dtype_include`, `dtype_exclude`, `pattern` |

Ils s'imbriquent : un `ColumnTransformer` contient des `Pipeline`, et se place lui-même comme
étape d'un `Pipeline` final.

### Les pièges rencontrés

| Piège | Symptôme | Parade |
|---|---|---|
| Catégorie stockée en nombre | `code_atelier` traité comme une grandeur continue | corriger le dtype en amont |
| Ordre alphabétique de `OrdinalEncoder` | `critique < majeure < mineure` | `categories=[ordre_metier]` explicite |
| Modalité inconnue en validation | exception pendant la CV | `handle_unknown="ignore"` |
| Scaler avant imputer | statistiques incohérentes | imputer d'abord, toujours |
| Preprocessing hors du pipeline | fuite, variable selon l'étape | tout mettre dedans |

### Les deux résultats mesurés

**La fuite par imputation est négligeable ici** ($10^{-4}$ d'AUC) ; la fuite par sélection de
features est catastrophique (0,88 au lieu de 0,50 sur du bruit pur). L'argument pour les
pipelines n'est pas la peur, c'est que la question ne se pose plus.

**Aucune des quatre variantes de preprocessing ne se distingue.** Quand les scores ne
départagent pas, choisissez le plus simple et le plus explicable.

### Auto-évaluation

Sans rouvrir le notebook :

1. Quelle est la différence de rôle entre `Pipeline` et `ColumnTransformer` ?
2. Pourquoi faut-il imputer **avant** de standardiser, et pas l'inverse ?
3. À quoi sert `handle_unknown="ignore"`, et que se passe-t-il sans lui ?
4. Que signifie le double underscore dans `prep__num__imputer__strategy` ?
5. Pourquoi `make_column_selector(dtype_include=np.number)` ne suffit-il pas toujours ?
6. Citez une étape de preprocessing où la fuite est négligeable, et une où elle est massive.
7. `MCAR`, `MAR` : qu'est-ce qui change dans la façon de traiter une colonne ?

### Rattachement au carnet

- **B1M-5** : mécanismes de manquance, stratégies d'imputation.
- **B1M-6** : variables catégorielles, ordinales, et le typage.
- **B1M-7** : fuite de données.
- **B1M-8** : pipelines.
- **B1M-9** : recherche d'hyperparamètres, y compris sur le preprocessing.
- **BHD-10** : modalité inconnue en production.
- **BHD-12** : des scores qui ne se départagent pas.